# 01 — Data Audit and Preparation

Audit of the NOAR extract and construction of the analytic dataset.

Outputs support Methodology Section 3.2 (Dataset) and Section 3.3 (Data Preparation).



In [ ]:
%pip install openpyxl scikit-learn


In [4]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 150)
pd.set_option('display.max_rows', 100)

RAW = '13_exact_conditions_bmi_height_weight.xlsx'
df = pd.read_excel(RAW)

print(f'File     : {RAW}')
print(f'Records  : {len(df):,}')
print(f'Patients : {df.regno.nunique():,}')
print(f'Variables: {df.shape[1]}')
print(f'Duplicate rows: {df.duplicated().sum()}')

File     : 13_exact_conditions_bmi_height_weight.xlsx
Records  : 33,356
Patients : 4,674
Variables: 46
Duplicate rows: 0


## A. Variable inventory

Every variable classified by role. 

In [5]:
ROLES = {
    'regno': 'identifier', 'fupno': 'identifier',
    'regyear': 'temporal', 'dobyear': 'temporal', 'follow_up_year': 'temporal',
    'ageons': 'demographic', 'agefu': 'demographic',
    'gender': 'demographic', 'ethnicity': 'demographic',
    'height': 'anthropometric', 'weight': 'anthropometric', 'bmi': 'anthropometric',
    'crpscore': 'das28_component', 'esr_score': 'das28_component',
    'swollen_28jt': 'das28_component', 'tend_28jt': 'das28_component',
    'das28': 'outcome_source', 'disease_activity': 'outcome',
    'Days_on_Steroids': 'treatment', 'Days_on_DMARDs': 'treatment',
    'Days_on_Biologics': 'treatment',
    'smokestatus': 'lifestyle', 'smokeday': 'lifestyle', 'health': 'lifestyle',
    'curemp': 'socioeconomic', 'occgrp': 'socioeconomic', 'sc': 'socioeconomic',
    'slfemp': 'socioeconomic', 'yrwork': 'socioeconomic',
    'died': 'status', 'yeardied': 'status',
    'dead_status': 'status', 'censored_status': 'status',
}

inv = pd.DataFrame({
    'variable': df.columns,
    'dtype': [str(df[x].dtype) for x in df.columns],
    'n_unique': [df[x].nunique() for x in df.columns],
    'pct_missing': [round(df[x].isna().mean() * 100, 2) for x in df.columns],
})
inv['role'] = inv.variable.map(ROLES).fillna(
    pd.Series(np.where(inv.variable.str.endswith('_status'), 'comorbidity', 'UNCLASSIFIED'),
              index=inv.index))

print(inv.sort_values(['role', 'variable']).to_string(index=False))
print()
print(inv.role.value_counts().to_string())
print()
unc = inv[inv.role == 'UNCLASSIFIED'].variable.tolist()
print(f'UNCLASSIFIED: {unc if unc else "none"}')
zv = [x for x in df.columns if df[x].nunique() <= 1]
print(f'Zero-variance: {zv if zv else "none"}')

         variable   dtype  n_unique  pct_missing            role
              bmi float64      4988        35.92  anthropometric
           height float64       145        35.75  anthropometric
           weight float64       765        35.90  anthropometric
       ang_status   int64         2         0.00     comorbidity
        bp_status   int64         2         0.00     comorbidity
    cancer_status   int64         2         0.00     comorbidity
   depress_status   int64         2         0.00     comorbidity
      diab_status   int64         2         0.00     comorbidity
      disc_status   int64         2         0.00     comorbidity
     glauc_status   int64         2         0.00     comorbidity
    htattk_status   int64         2         0.00     comorbidity
    htfail_status   int64         2         0.00     comorbidity
    kidney_status   int64         2         0.00     comorbidity
     liver_status   int64         2         0.00     comorbidity
      lung_status   int64

## B. Longitudinal structure

Establishes that the records represent genuine repeated measurement rather than repeated
copies, and quantifies the panel shape.

In [6]:
vpp = df.groupby('regno').size()
print(f'Visits per patient: mean {vpp.mean():.2f}  median {vpp.median():.0f}  '
      f'min {vpp.min()}  max {vpp.max()}')
print(f'Patients with >=2 visits: {(vpp >= 2).sum():,} ({(vpp >= 2).mean()*100:.1f}%)')

print('\nWithin-patient variation (missing values NOT counted as a level):')
rows = []
for x in ['agefu', 'weight', 'bmi', 'height', 'esr_score', 'crpscore',
          'swollen_28jt', 'tend_28jt', 'das28', 'disease_activity',
          'Days_on_Steroids', 'Days_on_DMARDs', 'Days_on_Biologics', 'smokestatus']:
    n   = df.groupby('regno')[x].nunique(dropna=True)
    obs = df[df[x].notna()].groupby('regno')[x].size()
    eligible = obs[obs > 1].index          # patients with >1 observed value
    rows.append({'variable': x,
                 'patients_with_>1_value': len(eligible),
                 'patients_varying': int((n.reindex(eligible) > 1).sum()),
                 'pct_of_eligible': round((n.reindex(eligible) > 1).mean() * 100, 1)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nThe denominator is patients with more than one observed value of that')
print('variable; patients who could not vary are excluded. Counting NaN as a level,')
print('or dividing by all patients, both distort the figure.')

print('\nAre treatment variables cumulative to the visit, or a study total?')
for x in ['Days_on_Steroids', 'Days_on_DMARDs', 'Days_on_Biologics']:
    mono = df.sort_values(['regno', 'fupno']).groupby('regno')[x].apply(
        lambda s: s.dropna().is_monotonic_increasing)
    print(f'  {x:20s} non-decreasing across visits: {mono.mean()*100:.1f}%')
print('  High values indicate exposure accumulated up to each visit, so the value at')
print('  visit t is legitimately available at the prediction point.')

Visits per patient: mean 7.14  median 7  min 2  max 15
Patients with >=2 visits: 4,674 (100.0%)

Within-patient variation (missing values NOT counted as a level):
         variable  patients_with_>1_value  patients_varying  pct_of_eligible
            agefu                    4674              4674            100.0
           weight                    3538              2636             74.5
              bmi                    3536              2663             75.3
           height                    3546              2501             70.5
        esr_score                    3218              1822             56.6
         crpscore                    2187              1489             68.1
     swollen_28jt                    4587              3820             83.3
        tend_28jt                    4587              4009             87.4
            das28                    2291              1879             82.0
 disease_activity                    2291              1339        

## C. Missingness

The extract retains its original missingness. This section quantifies the extent and
tests whether missingness in the outcome is related to calendar time, which determines
how temporal cohorts can be defined.

In [7]:
miss = (df.isna().mean() * 100).round(2)
miss = miss[miss > 0].sort_values(ascending=False)
print('Variables with missing data:')
print(miss.to_string())

print(f'\nComplete cases across all variables: {df.notna().all(axis=1).sum():,} '
      f'({df.notna().all(axis=1).mean()*100:.1f}%)')

print('\nOutcome missingness by calendar year of visit:')
by_yr = df.groupby(df.follow_up_year.round()).das28.agg(
    visits='size', labelled=lambda s: s.notna().sum())
by_yr['pct_labelled'] = (by_yr.labelled / by_yr.visits * 100).round(1)
print(by_yr.to_string())

Variables with missing data:
crpscore            67.15
das28               62.09
disease_activity    62.09
smokeday            56.27
health              50.27
esr_score           50.06
yrwork              43.74
bmi                 35.92
weight              35.90
height              35.75
smokestatus         22.84
sc                  20.05
slfemp              12.41
occgrp              10.32
ethnicity            3.59
curemp               2.28
follow_up_year       1.50
regyear              1.50
ageons               1.48
dobyear              1.41
gender               1.38
Days_on_Steroids     1.32
tend_28jt            1.29
swollen_28jt         1.29

Complete cases across all variables: 1,711 (5.1%)

Outcome missingness by calendar year of visit:
                visits  labelled  pct_labelled
follow_up_year                                
1989.0              98         0           0.0
1990.0             357         0           0.0
1991.0             621         0           0.0
1992.0       

## D. Outcome validation

Tests whether disease_activity is a threshold recoding of DAS28, and whether DAS28
itself is a clinically computed value rather than a derived or imputed one.

In [8]:
lab = df.dropna(subset=['das28', 'disease_activity'])
print(f'Labelled visits: {len(lab):,} ({len(lab)/len(df)*100:.1f}%)  '
      f'patients: {lab.regno.nunique():,}')
print(f'das28 and disease_activity missing together: '
      f'{(df.das28.isna() == df.disease_activity.isna()).mean()*100:.2f}% of rows')

BANDS = [-np.inf, 2.6, 3.2, 5.1, np.inf]
NAMES = ['remission', 'low', 'medium', 'high']
recode = pd.cut(lab.das28, BANDS, labels=NAMES).astype(str)
print(f'\nAgreement with standard DAS28 thresholds: {(recode == lab.disease_activity).mean()*100:.2f}%')
print(lab.groupby('disease_activity').das28.agg(['min', 'max', 'count']).round(2).to_string())

comp = lab.dropna(subset=['tend_28jt', 'swollen_28jt', 'esr_score'])
formula = (0.56 * np.sqrt(comp.tend_28jt) + 0.28 * np.sqrt(comp.swollen_28jt)
           + 0.70 * np.log(comp.esr_score.clip(lower=1)))
print(f'\nDAS28 recomputed from its components (3-variable DAS28-ESR form, n={len(comp):,}):')
print(f'  correlation with recorded value: {comp.das28.corr(formula):.3f}')
print(f'  mean absolute difference        : {(comp.das28 - formula).abs().mean():.3f}')
print('  The strong relationship is consistent with the recorded DAS28 incorporating')
print('  these clinical components. The remaining difference is compatible with')
print('  information not present in the extract, such as patient global health.')

Labelled visits: 12,646 (37.9%)  patients: 2,291
das28 and disease_activity missing together: 100.00% of rows

Agreement with standard DAS28 thresholds: 100.00%
                  min   max  count
disease_activity                  
high              5.1  8.17   1892
low               2.6  3.20   2111
medium            3.2  5.10   4377
remission         0.0  2.60   4266

DAS28 recomputed from its components (3-variable DAS28-ESR form, n=12,646):
  correlation with recorded value: 0.941
  mean absolute difference        : 0.387
  The strong relationship is consistent with the recorded DAS28 incorporating
  these clinical components. The remaining difference is compatible with
  information not present in the extract, such as patient global health.


## D2. Data quality and cleaning

Values that are impossible on clinical grounds are identified and corrected under a
predefined rule. Treatment durations cannot be negative, so negative values are
data-entry errors and are set to zero. Counts are recorded before correction

In [9]:
print('Negative treatment durations before correction:')
for x in ['Days_on_Steroids', 'Days_on_DMARDs', 'Days_on_Biologics']:
    neg = df[x] < 0
    print(f'  {x:20s} {neg.sum():3d} values, {df.loc[neg, "regno"].nunique():2d} patients')

for x in ['Days_on_Steroids', 'Days_on_DMARDs', 'Days_on_Biologics']:
    n_neg = (df[x] < 0).sum()
    before = df[x].copy()
    df[x] = df[x].clip(lower=0)
    n_changed = ((before != df[x]) & before.notna()).sum()
    assert n_neg == n_changed, f'{x}: clip altered non-negative values'

print('\nCorrection applied; only negative values were altered.')
print('Cumulative treatment duration cannot be below zero, so these values are')
print('clinically invalid for this measure and were truncated to zero under a')
print('deterministic plausibility rule applied before any feature derivation.')
print('The source of the invalid values is not established by this audit.')

Negative treatment durations before correction:
  Days_on_Steroids      51 values, 10 patients
  Days_on_DMARDs        35 values,  7 patients
  Days_on_Biologics      4 values,  1 patients

Correction applied; only negative values were altered.
Cumulative treatment duration cannot be below zero, so these values are
clinically invalid for this measure and were truncated to zero under a
deterministic plausibility rule applied before any feature derivation.
The source of the invalid values is not established by this audit.


## E. Target leakage assessment

DAS28 is a formula over the tender and swollen joint counts, ESR and patient global
health. The outcome is a banding of DAS28. Any model given those components at the same
visit therefore reconstructs the outcome arithmetically rather than predicting it.

This section quantifies that, and provides the empirical justification for the
next-visit design.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

lab = lab.copy()
lab['y'] = lab.disease_activity.isin(['medium', 'high']).astype(int)

FEATURE_SETS = {
    'DAS28 alone':                    ['das28'],
    'components (TJC, SJC, ESR)':     ['tend_28jt', 'swollen_28jt', 'esr_score'],
    'components + CRP':               ['tend_28jt', 'swollen_28jt', 'esr_score', 'crpscore'],
    'no components':                  ['agefu', 'bmi', 'weight', 'Days_on_Steroids',
                                       'Days_on_DMARDs', 'smokestatus'],
}
out = []
for name, cols in FEATURE_SETS.items():
    sub = lab.dropna(subset=cols)
    auc = cross_val_score(
        RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
        sub[cols], sub.y, groups=sub.regno, cv=GroupKFold(3), scoring='roc_auc')
    out.append({'feature_set': name, 'n': len(sub), 'AUC': round(auc.mean(), 3)})

print('Same-visit classification, GroupKFold by patient:')
print(pd.DataFrame(out).to_string(index=False))
print('\nAn AUC near 1.0 from DAS28 alone is definitional, not predictive. The')
print('components alone approach it because DAS28 is a function of them.')

Same-visit classification, GroupKFold by patient:
               feature_set     n   AUC
               DAS28 alone 12646 1.000
components (TJC, SJC, ESR) 12646 0.952
          components + CRP 10595 0.946
             no components 11005 0.596

An AUC near 1.0 from DAS28 alone is definitional, not predictive. The
components alone approach it because DAS28 is a function of them.


## F. Temporal availability of the outcome

DAS28 was not recorded throughout the register's history. This section establishes when
recording began and how availability varies by registration period. The finding that
determines whether temporal cohorts can be anchored on registration year.

In [11]:
print('Labelled visits by calendar year:')
print(lab.follow_up_year.round().value_counts().sort_index().to_string())

pat = df.groupby('regno').agg(reg=('regyear', 'first'),
                              any_das=('das28', lambda s: s.notna().any())).dropna(subset=['reg'])
pat['band'] = pd.cut(pat.reg, [-np.inf, 1994, 1999, 2004, np.inf],
                     labels=['<1995', '1995-1999', '2000-2004', '2005+'])
g = pat.groupby('band', observed=True).agg(patients=('any_das', 'size'),
                                           with_das=('any_das', 'sum'))
g['pct'] = (g.with_das / g.patients * 100).round(1)
print('\nOutcome availability by registration period:')
print(g.to_string())

lab2 = lab.merge(pat[['reg']], left_on='regno', right_index=True, how='left')
lab2['duration'] = lab2.agefu - lab2.ageons
lab2['band'] = pd.cut(lab2.reg, [-np.inf, 1994, 1999, 2004, np.inf],
                      labels=['<1995', '1995-1999', '2000-2004', '2005+'])
print('\nDisease duration (years) at the labelled visit, by registration period:')
print(lab2.groupby('band', observed=True).duration.agg(['count', 'mean', 'median']).round(1).to_string())
print('\nDivergent durations indicate registration period is confounded with disease')
print('stage among labelled observations.')

Labelled visits by calendar year:
follow_up_year
2001.0       1
2002.0       4
2003.0      12
2004.0     247
2005.0     773
2006.0     963
2007.0    1132
2008.0    1187
2009.0    1058
2010.0    1099
2011.0    1099
2012.0    1134
2013.0    1059
2014.0    1049
2015.0    1008
2016.0     612
2017.0     150

Outcome availability by registration period:
           patients  with_das   pct
band                               
<1995          1605       390  24.3
1995-1999       930         1   0.1
2000-2004       794       660  83.1
2005+          1239      1222  98.6

Disease duration (years) at the labelled visit, by registration period:
           count  mean  median
band                          
<1995       1389  19.6    20.0
1995-1999      3   7.7     8.0
2000-2004   4403   8.3     8.0
2005+       6773   5.6     5.0

Divergent durations indicate registration period is confounded with disease
stage among labelled observations.


## G. Construction of prediction pairs

Each pair consists of a predictor visit and the immediately following labelled visit.
Only information recorded at the predictor visit is retained; the outcome is the disease
activity category recorded at the following visit.

Pairs separated by more than two years are excluded, so that the prediction horizon is
approximately consistent.

In [12]:
d = df.sort_values(['regno', 'fupno']).copy()
d['y_next']    = d.groupby('regno').disease_activity.shift(-1)
d['das_next']  = d.groupby('regno').das28.shift(-1)
d['year_next'] = d.groupby('regno').follow_up_year.shift(-1)
d['gap']       = d.groupby('regno').agefu.shift(-1) - d.agefu

pairs_all = d[d.y_next.notna()].copy()
pairs = pairs_all.copy()
print(f'Pairs with a labelled following visit: {len(pairs):,}  '
      f'patients: {pairs.regno.nunique():,}')
print('\nPrediction horizon (years between visits):')
print(pairs.gap.value_counts().sort_index().to_string())

MAX_GAP = 2.0
pairs = pairs[pairs.gap <= MAX_GAP].copy()
print(f'\nAfter restricting to gap <= {MAX_GAP} years: {len(pairs):,} pairs, '
      f'{pairs.regno.nunique():,} patients')

n_within_horizon = len(pairs)
n_within_horizon_patients = pairs.regno.nunique()

testable = pairs[pairs.follow_up_year.notna() & pairs.dobyear.notna()]
agree = np.isclose(testable.follow_up_year, testable.dobyear + testable.agefu)
print(f'\nCalendar-year derivation check (dobyear + agefu):')
print(f'  pairs where both are available : {len(testable):,}')
print(f'  exact agreement                : {agree.sum():,} ({agree.mean()*100:.2f}%)')
assert agree.all(), 'derivation does not reproduce the recorded calendar year'

pairs['follow_up_year_original'] = pairs.follow_up_year
missing_before = pairs.follow_up_year.isna().sum()
pairs['follow_up_year'] = pairs.follow_up_year.fillna(pairs.dobyear + pairs.agefu)
missing_after = pairs.follow_up_year.isna().sum()
n_recovered = missing_before - missing_after

print(f'Calendar year missing originally : {missing_before}')
print(f'Recovered from dobyear + agefu   : {missing_before - missing_after}')
print(f'Unrecoverable, excluded          : {missing_after}')

excluded = pairs[pairs.follow_up_year.isna()]
print(f'  excluded pairs active rate     : '
      f'{excluded.y_next.isin(["medium","high"]).mean()*100:.1f}%')

pairs = pairs[pairs.follow_up_year.notna()].copy()
print(f'\nPairs with assignable calendar year: {len(pairs):,} '
      f'({pairs.regno.nunique():,} patients)')

pairs['y'] = pairs.y_next.isin(['medium', 'high']).astype(int)
print(f'Outcome balance: {pairs.y.sum():,} active ({pairs.y.mean()*100:.1f}%) / '
      f'{(1-pairs.y).sum():,} inactive')
print(f'Pairs per patient: median {pairs.groupby("regno").size().median():.0f}')
print('\nNote: the predictor visit need not itself be labelled, so pairs are available')
print('wherever the following visit has a recorded outcome.')

Pairs with a labelled following visit: 11,279  patients: 2,291

Prediction horizon (years between visits):
gap
0.5      123
1.0     7901
1.5       56
2.0     2277
2.5       44
3.0      613
3.5       17
4.0        3
4.5       17
5.0      207
5.5        8
6.0        1
6.5        6
7.0        2
8.5        3
11.0       1

After restricting to gap <= 2.0 years: 10,357 pairs, 2,291 patients

Calendar-year derivation check (dobyear + agefu):
  pairs where both are available : 10,306
  exact agreement                : 10,306 (100.00%)
Calendar year missing originally : 51
Recovered from dobyear + agefu   : 17
Unrecoverable, excluded          : 34
  excluded pairs active rate     : 17.6%

Pairs with assignable calendar year: 10,323 (2,279 patients)
Outcome balance: 4,877 active (47.2%) / 5,446 inactive
Pairs per patient: median 4

Note: the predictor visit need not itself be labelled, so pairs are available
wherever the following visit has a recorded outcome.


## H. Temporal cohort options

Cohorts are anchored on the calendar year of the predictor visit, since that is the
point at which the prediction is made and therefore the clinical environment the model
operates in.

In [13]:
# Assign the chosen three-cohort scheme
pairs['cohort'] = pd.cut(pairs.follow_up_year, [-np.inf, 2007, 2011, np.inf],
                         labels=['<=2007', '2008-2011', '2012+'])

g = pairs.groupby('cohort', observed=True).agg(
    pairs=('y', 'size'), patients=('regno', 'nunique'), pct_active=('y', 'mean'))
g['pct_active'] = (g.pct_active * 100).round(1)
g['mean_duration'] = pairs.assign(dur=pairs.agefu - pairs.ageons).groupby(
    'cohort', observed=True).dur.mean().round(1)
print(g.to_string())

counts = pairs.cohort.value_counts().sort_index()
print(f'\nsum: {counts.sum():,}  |  total pairs: {len(pairs):,}')
assert pairs.cohort.notna().all(), 'unassigned cohort rows remain'
assert counts.sum() == len(pairs), 'cohort counts do not sum to total'

spans = pairs.groupby('regno').cohort.nunique()
print(f'patients appearing in >1 cohort: {(spans > 1).sum():,} '
      f'({(spans > 1).mean()*100:.1f}%)')
print('Patients spanning cohorts require grouped cross-validation.')

           pairs  patients  pct_active  mean_duration
cohort                                               
<=2007      3364      1199        54.8            4.2
2008-2011   3888      1680        48.0            7.1
2012+       3071      1680        38.0            9.2

sum: 10,323  |  total pairs: 10,323
patients appearing in >1 cohort: 1,499 (65.8%)
Patients spanning cohorts require grouped cross-validation.


## H2. Leakage audit of status variables

The mortality and censoring variables are examined for future information. A variable
that is informative about whether a subsequent visit exists is derived from the future
relative to the prediction point, and cannot be used as a predictor.

In [14]:
d['has_next'] = d.y_next.notna()

print('Association between status variables and the existence of a labelled next visit:')
rows = []
for x in ['censored_status', 'dead_status']:
    ct = pd.crosstab(d[x], d.has_next, normalize='index')
    for lvl in ct.index:
        rows.append({'variable': x, 'value': lvl,
                     'pct_with_labelled_next_visit': round(ct.loc[lvl, True] * 100, 1)})
print(pd.DataFrame(rows).to_string(index=False))

print(f'\ndied varies within patient: '
      f'{(d.groupby("regno").died.nunique(dropna=True) > 1).sum()} patients')
print(f'yeardied populated on     : {d.yeardied.notna().sum():,} of {len(d):,} rows '
      f'({d.loc[d.yeardied.notna(), "regno"].nunique():,} patients)')

print('\nAll four status variables are excluded from the predictor set: they are either')
print('recorded on every record regardless of when the event occurred, or they are')
print('informative about whether a subsequent visit exists.')

Association between status variables and the existence of a labelled next visit:
       variable  value  pct_with_labelled_next_visit
censored_status      0                          38.2
censored_status      1                           5.3
    dead_status      0                          35.0
    dead_status      1                           0.0

died varies within patient: 0 patients
yeardied populated on     : 33,356 of 33,356 rows (4,674 patients)

All four status variables are excluded from the predictor set: they are either
recorded on every record regardless of when the event occurred, or they are
informative about whether a subsequent visit exists.


## H3. Derived predictors

Two variables are derived from information available at the predictor visit. Disease
duration is retained because it differs systematically between cohorts and must be
adjusted for rather than left implicit.

In [15]:
pairs['disease_duration'] = pairs.agefu - pairs.ageons

COMORB = [x for x in df.columns
          if x.endswith('_status') and x not in ('censored_status', 'dead_status')]
pairs['comorbidity_count'] = pairs[COMORB].sum(axis=1)

pairs['biologic_exposed'] = (pairs.Days_on_Biologics.fillna(0) > 0).astype(int)

# Audit: are the comorbidity indicators incident, or stamped retrospectively?
_s = df.sort_values(['regno', 'fupno'])
_rows = []
for x in COMORB:
    mono = _s.groupby('regno')[x].apply(lambda v: v.dropna().is_monotonic_increasing)
    ever = _s.groupby('regno')[x].max()
    pos = ever[ever == 1].index
    first = _s[_s.regno.isin(pos)].groupby('regno')[x].first()
    _rows.append({'indicator': x, 'pct_patients_non_decreasing': round(mono.mean()*100, 1),
                  'ever_positive_patients': len(pos),
                  'pct_positive_at_first_visit': round(first.mean()*100, 1)})
_aud = pd.DataFrame(_rows).sort_values('pct_positive_at_first_visit')
print('\nComorbidity indicator audit:')
print(_aud.to_string(index=False))
print(f'\nAll {len(COMORB)} indicators non-decreasing within patient: '
      f'{(_aud.pct_patients_non_decreasing == 100).all()}')
print('Many patients who are ever positive are not positive at their first recorded')
print('visit, supporting interpretation as diagnoses accumulated by the visit rather')
print('than retrospectively applied study-level indicators.')

print(f'disease_duration  : median {pairs.disease_duration.median():.1f} years, '
      f'range {pairs.disease_duration.min():.0f}-{pairs.disease_duration.max():.0f}')
print(f'comorbidity_count : {pairs.comorbidity_count.value_counts().sort_index().to_dict()}')
print(f'biologic_exposed  : {pairs.biologic_exposed.sum():,} pairs '
      f'({pairs.biologic_exposed.mean()*100:.2f}%), '
      f'{pairs.loc[pairs.biologic_exposed == 1, "regno"].nunique():,} patients')

print(f'\npairs: {pairs.shape[0]:,} rows, {pairs.shape[1]} columns')


Comorbidity indicator audit:
     indicator  pct_patients_non_decreasing  ever_positive_patients  pct_positive_at_first_visit
   disc_status                        100.0                     107                          0.0
 htfail_status                        100.0                      48                         12.5
 stroke_status                        100.0                      78                         20.5
 kidney_status                        100.0                      33                         21.2
 htattk_status                        100.0                     131                         26.7
    ang_status                        100.0                     164                         27.4
  glauc_status                        100.0                      91                         29.7
  liver_status                        100.0                      46                         30.4
   diab_status                        100.0                     256                         38.3


## H4. Feature eligibility

Every variable is classified as predictor, target or excluded, with the rationale
recorded. Missingness and cohort-specific availability are computed from the
prediction pairs rather than the full extract, since only the pairs enter modelling.

In [16]:
pairs['cohort'] = pd.cut(pairs.follow_up_year, [-np.inf, 2007, 2011, np.inf],
                         labels=['<=2007', '2008-2011', '2012+'])

ELIG = [
    ('agefu',             'predictor', 'Yes', 'Age at predictor visit',                            'time-varying'),
    ('gender',            'predictor', 'Yes', 'Baseline characteristic',                           'baseline'),
    ('disease_duration',  'predictor', 'Yes', 'Derived: agefu - ageons',                           'time-varying'),
    ('bmi',               'predictor', 'Yes', 'Standardised anthropometric measure',               'time-varying'),
    ('tend_28jt',         'predictor', 'Yes', 'Prior clinical state',                              'time-varying'),
    ('swollen_28jt',      'predictor', 'Yes', 'Prior clinical state',                              'time-varying'),
    ('esr_score',         'predictor', 'Yes', 'Prior clinical state',                              'time-varying'),
    ('Days_on_Steroids',  'predictor', 'Yes', 'Cumulative exposure to visit',                      'time-varying'),
    ('Days_on_DMARDs',    'predictor', 'Yes', 'Cumulative exposure to visit',                      'time-varying'),
    ('biologic_exposed',  'predictor', 'Yes', 'Binary: continuous measure zero for 93.6% of pairs','time-varying'),
    ('comorbidity_count', 'predictor', 'Yes', 'Diagnosed burden by visit',                         'time-varying'),
    ('sc',                'predictor', 'Yes', 'Socioeconomic classification',                      'baseline'),
    ('y',                 'target',    'n/a', 'Disease activity at following visit',               '-'),
    ('ethnicity',         'excluded',  'Yes', 'Retained for descriptive reporting only; too few observations in the smaller categories to support reliable cohort-specific estimation', 'baseline'),
    ('smokestatus',       'excluded',  'Yes', 'Largely invariant within patient (2.5% changed category among patients with more than one recorded value) and 13.3% missing; contemporaneous status at each predictor visit cannot be established reliably', '-'),
    ('das28',             'excluded',  'Yes', 'Composite of retained joint-count and inflammatory-marker variables; excluded to preserve granular feature attribution and avoid explanation dominance by a composite score', '-'),
    ('crpscore',          'excluded',  'Yes', 'Differential availability across cohorts',          '-'),
    ('Days_on_Biologics', 'excluded',  'Yes', 'Replaced by binary indicator',                      '-'),
    ('height',            'excluded',  'Yes', 'Redundant with BMI',                                '-'),
    ('weight',            'excluded',  'Yes', 'Redundant with BMI',                                '-'),
    ('disease_activity',  'excluded',  'Yes', 'Categorical derivative of predictor-visit DAS28; excluded to avoid duplicating the same composite clinical state in coarsened form', '-'),
    ('died',              'excluded',  'No',  'Future information',                                '-'),
    ('yeardied',          'excluded',  'No',  'Recorded on all rows regardless of date',           '-'),
    ('dead_status',       'excluded',  'No',  'Determines whether a next visit exists',            '-'),
    ('censored_status',   'excluded',  'No',  'Determines whether a next visit exists',            '-'),
    ('regno',             'excluded',  'Yes', 'Identifier; retained for grouped validation',       '-'),
    ('fupno',             'excluded',  'Yes', 'Identifier',                                        '-'),
    ('follow_up_year',    'excluded',  'Yes', 'Defines cohort membership',                         '-'),
    ('regyear',           'excluded',  'Yes', 'Proxy for cohort membership',                       '-'),
]

rows = []
for v, role, avail_t, reason, tv in ELIG:
    if v in pairs.columns:
        pct = round(pairs[v].isna().mean() * 100, 1)
        bc = pairs.groupby('cohort', observed=True)[v].apply(
            lambda s: round(s.notna().mean() * 100, 1)).to_dict()
        av = ' / '.join(str(x) for x in bc.values())
    else:
        pct, av = np.nan, '-'
    rows.append({'variable': v, 'role': role, 'available_at_t': avail_t,
                 'pct_missing': pct, 'availability_by_cohort_%': av,
                 'type': tv, 'rationale': reason})

elig = pd.DataFrame(rows)
print(elig.to_string(index=False))
print(f'\npredictors: {(elig.role == "predictor").sum()}  |  '
      f'excluded: {(elig.role == "excluded").sum()}')

elig.to_csv('methodology_feature_eligibility.csv', index=False)
print('Saved methodology_feature_eligibility.csv  (Appendix Table B2)')

         variable      role available_at_t  pct_missing availability_by_cohort_%         type                                                                                                                                                                                                  rationale
            agefu predictor            Yes          0.0    100.0 / 100.0 / 100.0 time-varying                                                                                                                                                                                     Age at predictor visit
           gender predictor            Yes          0.0    100.0 / 100.0 / 100.0     baseline                                                                                                                                                                                    Baseline characteristic
 disease_duration predictor            Yes          0.1     100.0 / 100.0 / 99.5 time-varying                

In [17]:
n_records            = len(df)
n_labelled           = df.disease_activity.notna().sum()
n_labelled_patients  = df.loc[df.disease_activity.notna(), 'regno'].nunique()

print('Sample construction')
print(f'  Records in extract                       : {n_records:,}  '
      f'({df.regno.nunique():,} patients)')
print(f'  Visits with recorded disease activity    : {n_labelled:,}  '
      f'({n_labelled_patients:,} patients)')
print(f'  Pairs with a labelled following visit    : {len(pairs_all):,}  '
      f'({pairs_all.regno.nunique():,} patients)')
print(f'  Pairs within a two-year horizon          : {n_within_horizon:,}  '
      f'({n_within_horizon_patients:,} patients)')
print(f'  Pairs with assignable calendar year      : {len(pairs):,}  '
      f'({pairs.regno.nunique():,} patients)')

PRED = [v for v, r, _, _, _ in ELIG if r == 'predictor']
cc = pairs.dropna(subset=[p for p in PRED if p in pairs.columns])
print(f'  Complete on all {len(PRED)} predictors           : {len(cc):,}  '
      f'({cc.regno.nunique():,} patients)')


# Table 3.1
table_3_1 = pd.DataFrame([
    ('Records in supplied NOAR extract', n_records, df.regno.nunique(),
     'Initial dataset'),
    ('Visits with recorded disease activity', int(n_labelled), n_labelled_patients,
     'Outcome measure available'),
    ('Candidate prediction pairs', len(pairs_all), pairs_all.regno.nunique(),
     'Labelled outcome visit had an available preceding predictor visit'),
    (f'Pairs within a {MAX_GAP:.0f}-year prediction horizon',
     n_within_horizon, n_within_horizon_patients,
     f'{len(pairs_all) - n_within_horizon:,} pairs with longer intervals excluded'),
    ('Analytic sample (predictor-visit year assigned)', len(pairs),
     pairs.regno.nunique(),
     f'{n_recovered} years recovered by derivation; '
     f'{missing_after} pairs excluded because calendar year was not derivable'),
], columns=['Stage', 'Records or pairs', 'Patients', 'Reason for reduction'])

table_3_1.to_csv('table_3_1_sample_construction.csv', index=False)
print('\nMethodology Table 3.1 - Sample construction')
display(table_3_1)


#  Table 3.2
earliest_patients = set(pairs.loc[pairs.cohort == '<=2007', 'regno'])
rows = []
for c in ['<=2007', '2008-2011', '2012+']:
    g = pairs[pairs.cohort == c]
    n = g if c == '<=2007' else g[~g.regno.isin(earliest_patients)]
    rows.append((c, len(g), g.regno.nunique(), round(g.y.mean() * 100, 1),
                 np.nan if c == '<=2007' else len(n),
                 np.nan if c == '<=2007' else n.regno.nunique(),
                 np.nan if c == '<=2007' else round(n.y.mean() * 100, 1)))

table_3_2 = pd.DataFrame(rows, columns=[
    'Temporal cohort', 'Pairs (full)', 'Patients (full)', 'Active % (full)',
    'Pairs (patient-disjoint)', 'Patients (patient-disjoint)',
    'Active % (patient-disjoint)'])

table_3_2.to_csv('table_3_2_population_architecture.csv', index=False)
print('Methodology Table 3.2 - Population architecture')
print('(patient-disjoint columns are not applicable to the earliest cohort)')
display(table_3_2)


# ---------------------------------- complete-case retention by evaluation population
print('\nComplete-case retention on the 12 predictors, by evaluation population:')
_pops = [('Earliest cohort (<=2007)', pairs[pairs.cohort == '<=2007'])]
for c in ['2008-2011', '2012+']:
    g = pairs[pairs.cohort == c]
    _pops.append((f'{c} patient-disjoint', g[~g.regno.isin(earliest_patients)]))
for lab_, s_ in _pops:
    k = s_.dropna(subset=[p for p in PRED if p in s_.columns])
    print(f'  {lab_:28s} {len(k):5,} of {len(s_):5,} ({len(k)/len(s_)*100:5.1f}%), '
          f'{len(s_) - len(k):4,} lost')
print('Complete-case exclusion would remove a much larger share of the earliest period,')
print('which is the era used for model development, so imputation is preferred as the')
print('primary strategy to avoid period-dependent selection.')


Sample construction
  Records in extract                       : 33,356  (4,674 patients)
  Visits with recorded disease activity    : 12,646  (2,291 patients)
  Pairs with a labelled following visit    : 11,279  (2,291 patients)
  Pairs within a two-year horizon          : 10,357  (2,291 patients)
  Pairs with assignable calendar year      : 10,323  (2,279 patients)
  Complete on all 12 predictors           : 9,370  (2,169 patients)

Methodology Table 3.1 - Sample construction


,Stage,Records or pairs,Patients,Reason for reduction
0,Records in supplied NOAR extract,33356,4674,Initial dataset
1,Visits with recorded disease activity,12646,2291,Outcome measure available
2,Candidate prediction pairs,11279,2291,Labelled outcome visit had an available preced...
3,Pairs within a 2-year prediction horizon,10357,2291,922 pairs with longer intervals excluded
4,Analytic sample (predictor-visit year assigned),10323,2279,17 years recovered by derivation; 34 pairs exc...


Methodology Table 3.2 - Population architecture
(patient-disjoint columns are not applicable to the earliest cohort)


,Temporal cohort,Pairs (full),Patients (full),Active % (full),Pairs (patient-disjoint),Patients (patient-disjoint),Active % (patient-disjoint)
0,<=2007,3364,1199,54.8,NaN,NaN,NaN
1,2008-2011,3888,1680,48.0,1208.0,621.0,47.2
2,2012+,3071,1680,38.0,1641.0,884.0,35.6



Complete-case retention on the 12 predictors, by evaluation population:
  Earliest cohort (<=2007)     2,777 of 3,364 ( 82.6%),  587 lost
  2008-2011 patient-disjoint   1,140 of 1,208 ( 94.4%),   68 lost
  2012+ patient-disjoint       1,511 of 1,641 ( 92.1%),  130 lost
Complete-case exclusion would remove a much larger share of the earliest period,
which is the era used for model development, so imputation is preferred as the
primary strategy to avoid period-dependent selection.


## I. Save

The pair-level dataset is saved before feature selection and missing-data handling, so
that those decisions can be revisited without rebuilding the pairs.

In [18]:
OUT = 'noar_prediction_pairs.csv'
pairs.to_csv(OUT, index=False)
print(f'Saved {OUT}  ({len(pairs):,} pairs, {pairs.shape[1]} columns)')

inv.to_csv('methodology_variable_inventory.csv', index=False)
print('Saved methodology_variable_inventory.csv  (basis for Appendix Table B1)')

Saved noar_prediction_pairs.csv  (10,323 pairs, 56 columns)
Saved methodology_variable_inventory.csv  (basis for Appendix Table B1)
